# pii

> whether a document is somebody's business, decided by arithmetic rather than by a model

In [ ]:
#| default_exp pii

PII detection uses checksummed patterns read in context: 0.998 precision at recall 1.000
on 720 generated documents and 0 false positives on 2.3 M characters of real legislation
(`evals/pii.py`, `evals/pii_real.py`). API keys (`secret`) gate too. Modes live on `ask(pii=…)`: see [ask](02_ask.ipynb).


In [ ]:
#| export
import re, json
from pathlib import Path
from fastcore.all import AttrDict, L


## What counts

- `MAX_SCAN`: sample both ends of a long document (headers hold account numbers).
- `DENSE`: matches per thousand chars; for callers separating a signature block from a customer list, not `has_pii`.
- `NER_CHARS` (20,000, from `extract`): cap on the opt-in name pass (`ner=True`).


In [ ]:
#| export
MAX_SCAN, DENSE = 200_000, 1.0

In [ ]:
#| export
def luhn(s:str) -> bool:
    "The check digit every payment card carries. Sixteen digits that fail it are not a card."
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) < 12: return False
    tot, parity = 0, len(ds) % 2
    for i, d in enumerate(ds):
        if i % 2 == parity: d *= 2; d -= 9 if d > 9 else 0
        tot += d
    return tot % 10 == 0

def _iban_ok(s:str) -> bool:
    "IBAN's mod-97 check: move the country prefix to the end, letters to digits, remainder must be 1."
    s = re.sub(r'[^A-Za-z0-9]', '', s).upper()
    if not (15 <= len(s) <= 34): return False
    t = s[4:] + s[:4]
    try: n = int(''.join(str(int(c, 36)) for c in t))
    except ValueError: return False
    return n % 97 == 1

def _nhs_ok(s:str) -> bool:
    "The UK NHS number's mod-11 check digit. Ten digits in a row are otherwise just ten digits."
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 10: return False
    tot = sum(d * (10 - i) for i, d in enumerate(ds[:9]))
    chk = 11 - tot % 11
    return chk != 10 and (0 if chk == 11 else chk) == ds[9]

def _ssn_ok(s:str) -> bool:
    "A US SSN's structurally impossible cases, which is as much as arithmetic can say about one."
    ds = re.sub(r'\D', '', s)
    if len(ds) != 9: return False
    a, b, c = ds[:3], ds[3:5], ds[5:]
    return a not in ('000', '666') and a[0] != '9' and b != '00' and c != '0000'

Checksums lift precision from 0.970 to 0.998 at recall 1.000 on 720 documents half carrying
lookalikes, over 8 draws (89/2880 → 7/2880 false positives, `evals/pii.py`). Every residual one is
a Luhn or mod-11 collision on a random digit run.


## What a number is not

Three things a checksum cannot see. A standards designation has the shape of a US ZIP (`EN 60601`),
a ten-digit path segment in a URL passes the NHS mod-11 check about one time in eleven
(`.../pages/2377744435`), and a slice of a space-grouped money amount has the shape of a phone
number (`EUR 360 000 000 000`). All three are decided by what sits around the digits.


In [ ]:
#| export
#: The fifty states plus DC. Two capitals before five digits are a ZIP only if they name a state.
US_STATES = frozenset(
    'AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM '
    'NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC'.split())
_ZIP = r'\b(?:' + '|'.join(sorted(US_STATES)) + r') \d{5}(?:-\d{4})?\b'

#: What names the digits after it as a reference rather than as somebody's. Matched against the
#: text to the left of a match, so `EN 60601` and `Order 4556737586899855` never reach a checksum.
#: The acronyms stay case-sensitive: lowercase `en` is an ordinary word. `No` is not here:
#: it costs `No. 5 Elm Street` and buys no precision (evals/pii.py).
DESIGNATOR = re.compile(
    r'(?:(?-i:\b(?:ISO|IEC|EN|BS|DIN|JIS|NZS|ASTM|ANSI|IEEE|NIST|NFPA|MIL|STD|RFC|ISBN|ISSN|DOI'
    r'|PMID|PMCID|CVE|CWE|GTIN|EAN|UPC|SKU|MPN)\b)'
    r'|\b(?:arxiv|figure|fig|table|section|clause|annex|appendix|exhibit|schedule|paragraph|para'
    r'|item|step|page|pages|line|row|column|col|footnote|volume|vol|chapter|part|version|ver'
    r'|build|revision|rev|commit|sha|release|port|ticket|issue|bug|order|invoice|receipt|docket'
    r'|serial|batch|lot|tracking|shipment|transaction|grid reference|ref)\b)'
    r'[\s.:#=/_-]{0,4}$', re.I)

#: Where the links are. Every digit run inside one is a path segment or a query value.
URLISH = re.compile(r'[a-z][a-z0-9+.-]*://\S+|\bwww\.\S+', re.I)
#: Kinds still worth finding inside a link: a key in a query string is still a key.
URL_KEEP = frozenset({'email', 'secret', 'ip'})

#: A digit group butting against a match, which makes the match a slice of one long space-grouped
#: number rather than a number of its own. One EU regulation is nine of these (evals/pii_real.py).
GROUP_L, GROUP_R = re.compile(r'\d[ \u00a0\u202f]$'), re.compile(r'^[ \u00a0\u202f]\d')

def _designated(s:str,  # the part being scanned
                i:int,  # where the match starts
) -> bool:
    "Is what starts at `i` introduced by a word that makes it a reference number?"
    return bool(DESIGNATOR.search(s[max(0, i - 48):i]))

def _grouped(s:str,  # the part being scanned
             i:int,  # where the match starts
             j:int,  # where the match ends
) -> bool:
    "Is `s[i:j]` a slice of a longer space-grouped digit run, as `000 000 000` is of `360 000 000 000`?"
    return bool(GROUP_L.search(s[max(0, i - 2):i]) or GROUP_R.match(s[j:j + 2]))


In [ ]:
#| export
#: kind -> (pattern, validator or None). Spans de-overlapped longest-first.
PATTERNS = {
    'email':   (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', None),
    # payment cards open 2-6 (Visa 4, Mastercard 2 and 5, Amex 3, Discover and UnionPay 6);
    # the other five leading digits are half of Luhn's collisions and none of its cards
    'card':    (r'\b[2-6](?:[ -]?\d){12,18}\b', luhn),
    'iban':    (r'\b[A-Z]{2}\d{2}[ ]?(?:[A-Z0-9]{4}[ ]?){2,7}[A-Z0-9]{1,4}\b', _iban_ok),
    'ssn':     (r'\b\d{3}-\d{2}-\d{4}\b', _ssn_ok),
    # grouped, or named: a bare ten-digit run passes mod-11 one time in eleven (evals/pii.py)
    'nhs':     (r'\b\d{3}[ -]\d{3}[ -]\d{4}\b'
                r'|\bnhs\s*(?:no|number|#)?\W{0,4}\d{3}[ -]?\d{3}[ -]?\d{4}\b', _nhs_ok),
    'phone':   (r'\+\d{1,3}[ .-]?\(?\d{1,5}\)?[ .-]?\d{3,4}[ .-]?\d{3,4}\b'
                r'|\(\d{2,5}\)[ .-]?\d{3,4}[ .-]?\d{3,4}\b'
                r'|\b0\d{1,4}[ .-]\d{3,4}[ .-]?\d{3,4}\b'
                r'|\b\d{3}-\d{3}-\d{4}\b'
                r'|\b(?:phone|tel|telephone|mobile|cell|fax)\b\W{0,8}\+?[\d ().-]{7,20}\d', None),
    'ip':      (r'\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b', None),
    'dob':     (r'\b(?:date of birth|dob|born)\b\W{0,12}(?:\d{1,4}[/-]\d{1,2}[/-]\d{1,4}|\d{1,2} \w+ \d{4})', None),
    # a cue followed by an ordinary word is a schema label, not a value: `passport, identity card`
    # and `driver's licence details` are both a lookahead away (evals/pii_real.py)
    'passport':(r'\b(?:passport(?:\s*(?:no|number|#))?)\W{0,6}(?=[A-Z0-9]{0,8}\d)[A-Z0-9]{6,9}\b', None),
    'licence': (r'\b(?:driver.?s? licen[cs]e|dl)(?:\s*(?:no|number|#))?\W{0,6}(?=[A-Z0-9]{0,19}\d)[A-Z0-9]{5,20}\b', None),
    'account': (r'\b(?:account|acct|a/c)(?:\s*(?:no|number|#))?\W{0,6}\d{6,17}\b', None),
    'sortcode':(r'\b(?:sort\s*code)\W{0,6}\d{2}[- ]?\d{2}[- ]?\d{2}\b', None),
    'secret':  (r'\b(?:sk-[A-Za-z0-9_-]{16,}|ghp_[A-Za-z0-9]{20,}|xox[baprs]-[A-Za-z0-9-]{10,}|AKIA[0-9A-Z]{16}|AIza[0-9A-Za-z_-]{35})\b', None),
    # likewise, and `patient name` is gone with it: a labelled field holds a name, which is
    # `person`, and `Medical record numbers;` is what a regulation about them says (evals/pii_real.py)
    'medical': (r'\b(?:patient\s*(?:id|no|number)|nhs\s*number|medical record(?:\s*(?:no|number|#))?|mrn)\b'
                r'\W{0,6}(?=[A-Za-z0-9-]{0,19}\d)[A-Za-z0-9-]{1,20}\b', None),
    # require a street name between number and suffix, and a named state before the ZIP. The
    # ZIP list is worth 0.998 vs 0.950; the street name is covered by DESIGNATOR on the corpus
    # in evals/pii.py and worth 0.965 vs 0.894 with that guard switched off.
    'address': (r'\b\d{1,5}[A-Za-z]?[ ,]+(?:[A-Z][A-Za-z.\'-]+[ ,]+){1,3}'
                r'(?:Street|St|Road|Rd|Avenue|Ave|Lane|Ln|Drive|Boulevard|Blvd|Close|Court|Ct'
                r'|Crescent|Way|Place|Terrace|Square|Sq|Gardens|Grove|Row|Walk)\b\.?'
                r'|\b[A-Z]{1,2}\d[A-Z\d]? ?\d[A-Z]{2}\b'
                r'|' + _ZIP, None),
}
#: `person` is the one kind arithmetic cannot find, and the only one gated behind `ner=True`.
IDENTIFYING = frozenset({'email', 'card', 'iban', 'ssn', 'nhs', 'phone', 'dob', 'passport', 'licence', 'account',
                         'sortcode', 'medical', 'secret', 'address', 'person',
                         'idnum', 'taxnum'})   # the two `MODEL_KINDS` no pattern has
CASED = frozenset({'address'})
_COMPILED = {k: (re.compile(p, 0 if k in CASED else re.I), v) for k, (p, v) in PATTERNS.items()}


## Where it is

In [ ]:
#| export
def _scan_parts(text:str, mx:int=MAX_SCAN) -> list:
    """Both ends of a long document as `(offset, part)` (headers/footers hold identifiers)."""
    text = str(text or '')
    if len(text) <= mx: return [(0, text)]
    half = mx // 2
    return [(0, text[:half]), (half + 1, text[-half:])]

def _scan_text(text:str, mx:int=MAX_SCAN) -> str:
    "The part of a long document worth scanning, joined for measurement. Offsets follow `_scan_parts`."
    return '\n'.join(p for _, p in _scan_parts(text, mx))

def person_spans(text:str,     # what to scan
                 mx:int=None,  # chars handed to the extractor; None -> `extract.NER_CHARS`
) -> L:
    """Honorific-anchored personal names. Off by default; costs an entity pass."""
    from vishalakshi.extract import NER_CHARS, _noun_ents
    text = str(text or '')[:mx or NER_CHARS]
    out = []
    for surface, label in _noun_ents(text):
        if label != 'PERSON': continue
        # `_noun_ents` collapses whitespace, so match across the line breaks a PDF leaves inside
        # a name -- and on word boundaries, or `Ross` masks the middle of `Rossini`.
        rx = r'\b' + r'\s+'.join(map(re.escape, surface.split())) + r'\b'
        out += [(m.start(), m.end(), 'person', m.group(0)) for m in re.finditer(rx, text)]
    return L(out)

def pii_spans(text:str,          # what to scan
              kinds=None,        # restrict to these kinds; None -> every pattern
              mx:int=MAX_SCAN,   # chars scanned before a long document is sampled at both ends
              ner:bool=False,    # also look for names, which no pattern can find
              model:bool=False,  # also run the classifier (`model_spans`), which costs weights
) -> L:
    """Every match, as `(start, end, kind, text)`, longest first and never overlapping."""
    want = set(kinds or (*_COMPILED, 'person'))
    mwant = want if kinds else MODEL_ADDS
    found = []
    for off, part in _scan_parts(text, mx):
        urls = [(m.start(), m.end()) for m in URLISH.finditer(part)]
        for kind, (rx, ok) in _COMPILED.items():
            if kind not in want: continue
            for m in rx.finditer(part):
                if ok is not None and not ok(m.group(0)): continue
                if kind not in URL_KEEP and any(a <= m.start() < b for a, b in urls): continue
                # only for a match that opens with a digit, so a cue-anchored one keeps its cue
                if m.group(0)[:1].isdigit() and (_designated(part, m.start())
                                                 or _grouped(part, m.start(), m.end())): continue
                found.append((off + m.start(), off + m.end(), kind, m.group(0)))
        if ner and 'person' in want:
            found += [(off + s, off + e, k, v) for s, e, k, v in person_spans(part)]
        if model:
            found += [(off + s, off + e, k, v) for s, e, k, v in model_spans(part) if k in mwant]
    found.sort(key=lambda s: (s[0] - s[1], s[0]))       # longest first, then leftmost
    out, taken = [], []
    for s, e, kind, val in found:
        if any(s < te and ts < e for ts, te in taken): continue
        taken.append((s, e))
        out.append((s, e, kind, val))
    return L(sorted(out))


In [ ]:
#| export
def pii_report(text:str,          # what to scan
               kinds=None,        # restrict to these kinds; None -> every pattern
               mx:int=MAX_SCAN,   # chars scanned before a long document is sampled at both ends
               ner:bool=False,    # also look for names
               model:bool=False,  # also run the classifier
) -> AttrDict:
    """Spans found and whether they tip `has_pii` (IDENTIFYING kinds only)."""
    spans, counts = pii_spans(text, kinds, mx, ner=ner, model=model), {}
    for _, _, k, _ in spans: counts[k] = counts.get(k, 0) + 1
    n = len(_scan_text(text, mx))
    ident = {k: v for k, v in counts.items() if k in IDENTIFYING}
    # `n` and `density` stay arithmetic-only, or `DENSE` would mean something new the day NER came on
    n_arith = len(spans) - counts.get('person', 0)
    return AttrDict(has_pii=bool(ident), kinds=counts, identifying=ident, n=n_arith,
                    n_person=counts.get('person', 0), scanned=n, scanned_ner=bool(ner),
                    scanned_model=bool(model),
                    density=round(1000 * n_arith / max(n, 1), 3), spans=spans)


`has_pii` counts only `IDENTIFYING`. IP addresses remain reportable but do not trigger the gate.


In [ ]:
#| export
def redact(text:str,       # the text to mask
           spans=None,     # spans from `pii_spans`; recomputed over the whole of `text` when None
           kinds=None,     # restrict to these kinds
           mask:str=None,  # what to put in place of a match; None -> `[KIND]`
           ner:bool=False, # also mask names
           model:bool=False, # also mask what the classifier finds
) -> str:
    """Mask matched spans. Names only with `ner=True`."""
    out = str(text or '')
    if spans is None: spans = pii_spans(out, kinds, mx=len(out), ner=ner, model=model)
    for s, e, kind, _ in sorted(spans, reverse=True):
        out = out[:s] + (mask if mask is not None else f'[{kind.upper()}]') + out[e:]
    return out


## Asking the vault

`Vault.pii` is document-level. `pii_ctx` gates an answer over the sections retrieval chose.


## The model, if you want one

`model=True` runs a DeBERTa-v3 token classifier ([piiranha](https://huggingface.co/iiiorg/piiranha-v1-detect-personal-information),
ONNX) over the same text and returns spans in the same shape. Off by default and worth switching on
only for what the patterns cannot reach: names, and street lines with no number. It costs
`onnxruntime`, 1.1 GB of weights and 24 ms per thousand characters, and it misses email and IBAN
outright. Numbers in `evals/RESULTS.md`.

Use the fp32 build. The int8 one found 0 of 200 planted identities.


In [ ]:
#| export
#: The ONNX repo, and the build to take from it. `model_int8.onnx` is present and unusable.
PII_ONNX, PII_ONNX_FILE = 'onnx-community/piiranha-v1-detect-personal-information-ONNX', 'onnx/model.onnx'
MODEL_CHARS = 1400   #: chars per window, kept under the 512-token limit with room for long digit runs

#: model label -> our kind. `city` and `username` are reportable and do not gate: the model calls
#: `London` a city and the local part of an address a username.
MODEL_KINDS = {'I-EMAIL': 'email', 'I-CREDITCARDNUMBER': 'card', 'I-SOCIALNUM': 'ssn',
               'I-TELEPHONENUM': 'phone', 'I-DATEOFBIRTH': 'dob', 'I-ACCOUNTNUM': 'account',
               'I-IDCARDNUM': 'idnum', 'I-TAXNUM': 'taxnum', 'I-DRIVERLICENSENUM': 'licence',
               'I-PASSWORD': 'secret', 'I-STREET': 'address', 'I-BUILDINGNUM': 'address',
               'I-ZIPCODE': 'address', 'I-CITY': 'city', 'I-GIVENNAME': 'person',
               'I-SURNAME': 'person', 'I-USERNAME': 'username'}
#: What `model=True` takes from the classifier when `kinds` is not given: the one kind no pattern
#: reaches. Unioning everything it emits costs precision 0.996 -> 0.845 and buys no recall
#: (evals/pii_model.py). Pass `kinds` to ask for the rest.
MODEL_ADDS = frozenset({'person'})
_MODEL = {}

def _model(repo:str=None, fn:str=None):
    "The ONNX session, tokenizer and label table, loaded once. Downloads ~1.1 GB the first time."
    repo, fn = repo or PII_ONNX, fn or PII_ONNX_FILE
    if (repo, fn) in _MODEL: return _MODEL[(repo, fn)]
    try:
        import onnxruntime as ort
        from tokenizers import Tokenizer
        from huggingface_hub import hf_hub_download
    except ImportError as e:
        raise ImportError("pii(model=True) needs `pip install vishalakshi[model]`") from e
    cfg = json.loads(Path(hf_hub_download(repo, 'config.json')).read_text())
    tok = Tokenizer.from_file(hf_hub_download(repo, 'tokenizer.json'))
    sess = ort.InferenceSession(hf_hub_download(repo, fn),
                                providers=['CPUExecutionProvider'])
    labels = [cfg['id2label'][str(i)] for i in range(len(cfg['id2label']))]
    _MODEL[(repo, fn)] = (sess, tok, labels)
    return _MODEL[(repo, fn)]

def _windows(text:str, mx:int=MODEL_CHARS) -> list:
    "`(offset, chunk)` on line then space boundaries, so a span is never cut in half mid-number."
    out, i = [], 0
    while i < len(text):
        j = min(i + mx, len(text))
        if j < len(text):
            cut = max(text.rfind('\n', i + mx//2, j), text.rfind(' ', i + mx//2, j))
            if cut > i: j = cut
        out.append((i, text[i:j]))
        i = j if j > i else i + mx
    return out


In [ ]:
#| export
def model_spans(text:str,            # what to scan
                thresh:float=0.5,    # softmax floor for a token to count
                repo:str=None,       # ONNX repo; None -> `PII_ONNX`
                fn:str=None,         # file within it; None -> `PII_ONNX_FILE`
) -> L:
    "The classifier's spans, as `(start, end, kind, text)`. Adjacent tokens of one kind are merged."
    import numpy as np
    sess, tok, labels = _model(repo, fn)
    text, out = str(text or ''), []
    for off, chunk in _windows(text):
        enc = tok.encode(chunk)
        ids = np.asarray([enc.ids], dtype=np.int64)
        lg = sess.run(None, {'input_ids': ids, 'attention_mask': np.ones_like(ids)})[0][0]
        p = np.exp(lg - lg.max(-1, keepdims=True)); p /= p.sum(-1, keepdims=True)
        best, cur = p.argmax(-1), None
        for (a, b), i, row in zip(enc.offsets, best, p):
            if b <= a: continue                          # [CLS]/[SEP] carry an empty offset
            kind = MODEL_KINDS.get(labels[i]) if row[i] >= thresh else None
            a, b = off + a, off + b
            # one gap character absorbs the space or hyphen inside `4111 1111` and `Elm Street`
            if cur and kind == cur[2] and a - cur[1] <= 1: cur[1] = b
            else:
                if cur: out.append(tuple(cur))
                cur = [a, b, kind] if kind else None
        if cur: out.append(tuple(cur))
    return L([(a, b, k, text[a:b].strip()) for a, b, k in out if text[a:b].strip()])


In [ ]:
#| export
from vishalakshi.core import Vault
from fastcore.all import patch

def redact_obj(o, kinds=None, ner:bool=False, model:bool=False):
    "`redact` over the strings inside a nested dict or list: what a structured answer is."
    if isinstance(o, str):  return redact(o, kinds=kinds, ner=ner, model=model)
    if isinstance(o, dict): return {k: redact_obj(v, kinds, ner, model) for k, v in o.items()}
    if isinstance(o, list): return [redact_obj(v, kinds, ner, model) for v in o]
    return o

@patch
def pii(self:Vault,
        ref,                 # a doc_id, source, title or path: whatever `document` takes
        max_chars:int=MAX_SCAN,
        ner:bool=False,      # also look for names, except on code, where identifiers are not names
        model:bool=False,    # also run the classifier, likewise not on code
) -> AttrDict:
    "Whether one whole document is somebody's business, and what in it says so."
    d = self.document(ref, max_chars=max_chars)
    prose = (d.get('kind') or '') != 'code'
    # `scanned_ner` then reports False, which is the honest answer: nothing looked for a name here
    r = pii_report(d.text, ner=ner and prose, model=model and prose)
    override = (self.marks(d.get('doc_id')) or {}).get('pii_override') if d.get('doc_id') else None
    r.detected, r.override = r.has_pii, override
    if override == 'clear': r.has_pii = False
    elif override == 'force': r.has_pii = True
    r.doc_id, r.title, r.source = d.get('doc_id'), d.get('title'), d.get('source')
    return r

@patch
def mark_not_pii(self:Vault, ref, clear:bool=True, reason:str='') -> dict:
    "Clear a false-positive PII decision (`pii_override='clear'`), or restore automatic detection."
    return self.mark(ref, pii_override='clear' if clear else None,
                     pii_reason=(reason or None) if clear else None)

@patch
def mark_pii(self:Vault, ref, force:bool=True, reason:str='') -> dict:
    "Force a document private even when arithmetic finds nothing (names, addresses, whole PDFs)."
    return self.mark(ref, pii_override='force' if force else None,
                     pii_reason=(reason or None) if force else None)

def pii_ctx(ctx, ner:bool=False, model:bool=False) -> AttrDict:
    "The report for an assembled context, which is what a policy has to gate on."
    parts = [str(getattr(r, 'text', None) or (r.get('text') if isinstance(r, dict) else '') or '')
             for r in (list(ctx.get('results') or []) + list(ctx.get('related') or []))]
    return pii_report('\n\n'.join(parts), ner=ner, model=model)

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Using it

In [ ]:
report = pii_report("""
Invoice 4471 for Ada Lovelace <ada@example.com>, phone 020 7946 0958.
Card 4111 1111 1111 1111, sort code 20-00-00, account number 12345678.
Server 10.0.0.14 returned 500. Order number 4471000012345678.
""")
report.has_pii, report.identifying, report.n

(True, {'email': 1, 'phone': 1, 'card': 1, 'sortcode': 1, 'account': 1}, 6)

In [ ]:
from fastcore.test import test_eq
test_eq(report.has_pii, True)
test_eq('card' in report.identifying, True)      # passes Luhn
test_eq(report.kinds.get('ip'), 1)               # reported...
test_eq('ip' in report.identifying, False)       # ...but a log is not somebody's private life

In [ ]:
print(redact("""
Invoice 4471 for Ada Lovelace <ada@example.com>, phone 020 7946 0958.
Card 4111 1111 1111 1111, sort code 20-00-00.
""").strip())

Invoice 4471 for Ada Lovelace <[EMAIL]>, [PHONE].
Card [CARD], [SORTCODE].


Names are opt-in and honorific-anchored: `Dr Charles Babbage` matches, bare `Ada Lovelace` does
not. Read `scanned_ner` before a zero. `model=True` raises that 2/8 to 5/8 on sentences no pattern
reaches, at 73 ms per document. Street lines are arithmetic like the rest.


In [ ]:
#| hide
from fastcore.test import test_eq

# secrets gate: a key alone is identifying
test_eq(pii_report('export OPENAI_API_KEY=sk-abcdefghijklmnopqrstuvwxyz123456').has_pii, True)
test_eq('secret' in pii_report('token ghp_abcdefghijklmnopqrst').identifying, True)

# medical: research prose is not a medical record; a patient id is
test_eq(pii_report('This paper diagnoses a failure mode in the prescription of learning rates').has_pii, False)
test_eq(pii_report('Patient id 44291 was discharged yesterday.').has_pii, True)

# nested redact for structured answers
test_eq(redact_obj({'email': 'a@b.co', 'n': 1}), {'email': '[EMAIL]', 'n': 1})
test_eq(redact_obj(['a@b.co', 3])[0], '[EMAIL]')


# a street line is what makes "John Smith, 12 Elm Street" private: no pattern finds the name
test_eq(pii_report('John Smith, 12 Elm Street').identifying, {'address': 1})
test_eq(pii_report('900 Market St, San Francisco CA 94103').has_pii, True)
test_eq(pii_report('The registered office is 221B Baker Street, London NW1 6XE').kinds['address'], 2)
# a numbered heading is not an address, which is the whole precision argument
for _t in ('Chapter 4 Court decisions', 'Table 3 Road traffic figures', 'Figure 2 Way of working'):
    test_eq(pii_report(_t).has_pii, False)

In [ ]:
#| hide
# 1. off by default, capped at NER_CHARS when on
from vishalakshi.extract import NER_CHARS
_sig = 'Dr Charles Babbage signed it.'
test_eq(pii_report(_sig).has_pii, False)                       # names are not looked for
test_eq(pii_report(_sig, ner=True).identifying, {'person': 1})   # ...until asked
test_eq(pii_report('Ada Lovelace signed it.', ner=True).has_pii, False)   # an honorific is the anchor
test_eq(pii_report('x'*NER_CHARS + ' ' + _sig, ner=True).kinds.get('person'), None)   # past the cap

# 2. `scanned_ner` keeps "none found" apart from "not looked for"
test_eq(pii_report('nothing here').scanned_ner, False)
test_eq(pii_report('nothing here', ner=True).scanned_ner, True)

# names are masked once asked for, and not before
test_eq('[PERSON]' in redact(_sig), False)
test_eq(redact(_sig, ner=True), 'Dr [PERSON] signed it.')

# the seam: `_scan_parts` keeps the halves apart, so an honorific at the end of one and a
# capitalised pair at the start of the next is not a person who was never in the document
_half = 4000
_doc = 'x'*(_half-3) + ' Dr' + 'm'*5000 + 'Charles Babbage wrote it.' + 'z'*(_half-25)
test_eq(pii_report(_doc, mx=8000, ner=True).kinds.get('person'), None)

# `density` and `n` stay arithmetic-only, or `DENSE` would mean something new the day NER came on
_names = 'Dr Ada Lovelace met Dr Charles Babbage. '*5
test_eq((pii_report(_names).density, pii_report(_names, ner=True).density), (0.0, 0.0))
test_eq(pii_report(_names, ner=True).n_person, 10)

In [ ]:
#| hide
from tempfile import mkdtemp
from pathlib import Path
from vishalakshi import Vault

v = Vault(Path(mkdtemp())/'p.db', offline=True)
v.add('A letter about Jane, and what she said on Tuesday.', title='letter', source='/inbox/letter.md')
r = v.pii('/inbox/letter.md')
test_eq(r.has_pii, False)
v.mark_pii('/inbox/letter.md', reason='address book')
test_eq(v.pii('/inbox/letter.md').has_pii, True)
test_eq(v.pii('/inbox/letter.md').override, 'force')
v.mark_not_pii('/inbox/letter.md', reason='my own draft')
test_eq(v.pii('/inbox/letter.md').has_pii, False)
test_eq(v.pii('/inbox/letter.md').override, 'clear')

In [ ]:
#| hide
# 3. no NER on code: an identifier is not a name, and a report must not claim it looked
v.add('# Dr Charles Babbage wrote this\ndef f(): pass', title='mod', source='/m.py', kind='code')
_c = v.pii('/m.py', ner=True)
test_eq((_c.scanned_ner, _c.has_pii), (False, False))

v.add(_sig, title='signed', source='/inbox/signed.md')
_p = v.pii('/inbox/signed.md', ner=True)
test_eq((_p.scanned_ner, _p.has_pii, _p.identifying), (True, True, {'person': 1}))
test_eq(v.pii('/inbox/signed.md').scanned_ner, False)   # the default is still arithmetic only

A number that fails its checksum is not the thing the checksum protects.

In [ ]:
test_eq(pii_report('Order 4111 1111 1111 1112 shipped').has_pii, False)   # fails Luhn
test_eq(pii_report('Card 4111 1111 1111 1111 charged').has_pii, True)     # passes it
test_eq(pii_report('The build takes 20 minutes and costs nothing.').has_pii, False)

In [ ]:
#| hide
long_doc = 'Account number 12345678\n' + ('filler text. ' * 40_000) + '\nsigned, ada@example.com'
r = pii_report(long_doc)
test_eq(r.has_pii, True)
test_eq(sorted(r.identifying), ['account', 'email'])
test_eq(r.scanned <= MAX_SCAN + 1, True)

# ...but a *report* may sample and a redaction may not
masked = redact(long_doc)
assert 'ada@example.com' not in masked, masked[-80:]
assert '12345678' not in masked, masked[:80]
test_eq(masked.count('filler text. '), 40_000)      # and nothing in between was moved
test_eq(pii_report(masked).has_pii, False)

In [ ]:
#| hide
one = pii_report('4111 1111 1111 1111')
test_eq(one.n, 1)
test_eq(list(one.kinds), ['card'])

Phones need a separator or trunk prefix, cards need an issuer digit (0.998 vs 0.995 without) and
a ten-digit NHS number needs its groups or its name (0.998 vs 0.991). Bare digit runs are not
identity, and a digit group butting against one makes it a slice of a longer number rather than a
number (0.998 vs 0.935).


In [ ]:
#| hide
# What must and must not put a document on the local-only path
cases = [
    ('Order 4111 1111 1111 1112 shipped',                 False),   # fails Luhn
    ('Card 4111 1111 1111 1111 charged',                  True),
    ('phone 020 7946 0958',                               True),
    ('+44 20 7946 0958',                                  True),
    ('call (555) 123-4567',                               True),
    ('555-123-4567',                                      True),
    ('ada@example.com',                                   True),
    ('The build takes 20 minutes and costs nothing.',     False),
    ('Run 2024 1000 2000 3000 through the pipeline',      False),
    ('Release 1.2.3 shipped on 2024-05-01 with 400 tests', False),
    ('Server 10.0.0.14 returned 500',                     False),   # reported, not identifying
    ('commit 8f3a2b1 touched 120 lines in 14 files',      False),
]
for text, want in cases: test_eq((text, pii_report(text).has_pii), (text, want))

In [ ]:
#| hide
# The two failures this corpus was extended for. A standards designation has the shape of a US
# ZIP, and a ten-digit path segment passes the NHS mod-11 check about one time in eleven.
for _t in ('EN 60601-1 applies to medical electrical equipment.',
           'EN 60601-1-2 covers electromagnetic compatibility.',
           'Conformity with EN 60601 is required.',
           'BS EN 12345 was withdrawn.',
           'See https://example.atlassian.net/wiki/spaces/TA/pages/2377744435 for the spec.',
           'Order 2377744435 shipped.',
           'Page 2377744435 of the export was truncated.'):
    test_eq((_t, pii_report(_t).has_pii), (_t, False))

# ...without giving up the thing each guard sits next to
# the street line and the state+ZIP are two spans, and the ZIP needs the state to be one at all
test_eq(pii_report('Ship to 900 Market St, San Francisco CA 94103.').kinds, {'address': 2})
test_eq(pii_report('Ship to 900 Market St, San Francisco EN 94103.').kinds, {'address': 1})
test_eq(pii_report('NHS number 4505577104 recorded at triage.').has_pii, True)
test_eq(pii_report('Recorded at triage as 450 557 7104 on the day.').has_pii, True)
test_eq(pii_report('No. 5 Elm Street, London.').has_pii, True)   # `No.` numbers a house too
# a key in a query string is still a key, and an address in one is still an address
test_eq('secret' in pii_report('https://api.example.com/k?t=sk-abcdefghijklmnopqrstuvwxyz123456').identifying, True)
test_eq('email' in pii_report('https://x.example.com/f?to=jane@example.com').identifying, True)


In [ ]:
#| hide
# The three failures 2.3 M characters of real legislation found (evals/pii_real.py). A space is the
# thousands separator across Europe, and a cue word with no value after it is a schema label.
for _t in ('an amount of up to EUR 360 000 000 000 as referred to in point (b)',
           'EUR 6 074 000 000 in current prices referred to in point (a)',
           'a ceiling of EUR 1 000 000 000 shall not be exceeded',
           'entered as such in a passport, identity card or other document',
           "driver's licence details are verified at the counter",
           'Medical record numbers; Health plan beneficiary numbers;',
           'an entire medical record, except when specifically justified',
           'The patient name field is mandatory in the schema.'):
    test_eq((_t, pii_report(_t).has_pii), (_t, False))

# ...and every cue still reads a value when there is one
test_eq(pii_report('Passport number 512394872 was checked.').identifying, {'passport': 1})
test_eq(pii_report("Driver's licence D1234567 on file.").identifying, {'licence': 1})
test_eq(pii_report('Medical record number 4471882 was pulled.').identifying, {'medical': 1})
test_eq(pii_report('MRN 7741-22 in the chart.').identifying, {'medical': 1})
test_eq(pii_report('Call 020 7946 0958 before noon.').identifying, {'phone': 1})


In [ ]:
#| hide
# `model=True` adds `MODEL_ADDS` and nothing else: unioning every kind the classifier emits costs
# precision 0.996 -> 0.845 for no recall (evals/pii_model.py, on the 480-document corpus that
# preceded the regulatory pass). Skipped without onnxruntime.
try: import onnxruntime, tokenizers  # noqa: F401
except ImportError: pass
else:
    _m = pii_report('The claimant, Sarah Nakamura, disputes the figure entirely.', model=True)
    test_eq((_m.scanned_model, _m.identifying), (True, {'person': 1}))
    test_eq(pii_report('Order 4556737586899855 shipped on Tuesday.', model=True).has_pii, False)
    test_eq(pii_report('EN 60601-1 applies.', model=True).has_pii, False)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()